In [5]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [8]:
import os
import json
import random
import time
import pandas as pd

# import tqdm
# from pydantic import BaseModel
import tiktoken
from dotenv import load_dotenv
from openai import AzureOpenAI
from openai_cost_calculator import estimate_cost_typed
from typing import Any

load_dotenv(".env", verbose=True)
api_version = "2024-12-01-preview"
model_name = "gpt-4o"
deployment_name = "EstNLTK-gpt-4o"
endpoint = str(os.getenv("AZURE_ENDPOINT"))
api_key = str(os.getenv("OPENAI_API_KEY"))

### Testing


In [54]:
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=api_key,
)

In [51]:
# Estimate the cost of a request
max_tokens = 128
costs = {
    "gpt-4o": {
        "input": 2.50 / 1_000_000,  # $2.50 per 1M tokens
        "cached_input": 1.25 / 1_000_000,  # $1.25 per 1M tokens
        "output": 10.00 / 1_000_000,  # $10.00 per 1M tokens
    }
}

# 1) build messages
system_prompt = """You are an Estonian sentence rewriting assistant.

Replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Preserve tense, number, case, agreement, capitalisation, punctuation, and word order as much as possible.
- Infer the token’s grammatical role and morphology from the sentence context.
- Generate 10 alternatives that fit the context and use only these cases: nominative, genitive, partitive, or additive/illative (both fit the same role in this task).
- If the token is a proper name, replace it with another plausible proper name that still fits the required case.
- Do not repeat the original token unless it is the only valid option.
"""

# Structured output format for Azure OpenAI chat completions.
# Structured Outputs require a top-level object, so the list is wrapped in a candidates field.
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "candidates",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "candidates": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 10,
                    "maxItems": 10,
                }
            },
            "required": ["candidates"],
            "additionalProperties": False,
        },
    },
}

# One-shot example to anchor the output format
one_shot_user = """Sentence: Ma näen <maja>."""

one_shot_assistant = json.dumps(
    {
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "häärberit",
        ],
    },
    ensure_ascii=False,
    indent=2,
)

# User prompt example for the real sentence
user_prompt = """Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots."""
# user_prompt = """Provide 10 alternatives for the marked token in the following sentence: KOMISJONI MÄÄRUS (EMÜ) nr 344/86, millega muudetakse tagatiste osas määrusi (EMÜ) nr 626/85 ja (EMÜ) 2077/85 17. veebruar 1986 EUROOPA ÜHENDUSTE KOMISJON, võttes arvesse Euroopa Majandusühenduse asutamislepingut, võttes arvesse nõukogu 14. märtsi 1977. aasta määrust (EMÜ) nr 516/77 puu- ja köögiviljatooteturu ühise korralduse kohta, 1 viimati muudetud määrusega (EMÜ) nr 3768/85, 2 eriti selle artikli 4 lõiget 8, võttes arvesse nõukogu 14. märtsi 1977. aasta määrust (EMÜ) nr 525/77, millega kehtestatakse ananassikonservide tootmistoetuste süsteem, 3 viimati muudetud määrusega (EMÜ) nr 1699/85, 4 eriti selle artiklit 8, ning arvestades, et : komisjoni 22. juuli 1985. aasta määruse (EMÜ) nr 2220/85 (millega sätestatakse põllumajandustoodete tagatissüsteemi ühised üksikasjalikud rakenduseeskirjad) 5 III jaotises sätestatakse tagatiste andmise vormid ; sama määruse artikli 19 lõikes 1 sätestatakse seoses ettemaksetega esitatud tagatiste vabastamine ; sellest tulenevalt tuleks tunnistada kehtetuks komisjoni 12. märtsi 1985. aasta määruse (EMÜ) nr 626/85 (töötlemata kuivatatud viinamarjade ja viigimarjade ostmise, müümise ja ladustamise kohta ladustusasutuste poolt) 6 ja komisjoni 25. juuli 1985. aasta määruse (EMÜ) nr 2077/85 (milles sätestatakse ananassikonservide tootmistoetuse üksikasjalikud rakenduseeskirjad) 7 vastavad sätted ; käesoleva määrusega ettenähtud meetmed on kooskõlas puu- ja köögiviljatooteturu korralduskomitee arvamusega, ON VASTU VÕTNUD KÄESOLEVA MÄÄRUSE.""",
# user_prompt = """Väljakutsete ja abisaanute analüüsi alusel otsustatakse jaanuaris-veebruaris, kas Elvas hakkavad sõitma ka kogemustega Tartu kiirabi arstid-velskrid."""


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": one_shot_user},
    {"role": "assistant", "content": one_shot_assistant},
    {"role": "user", "content": user_prompt},
]

# 2) token count (use model encoding)
enc = tiktoken.encoding_for_model(model_name)  # pick same model you'll use
prompt_text = "".join(m["content"] for m in messages)
prompt_tokens = len(enc.encode(prompt_text))
print(f"Prompt tokens: {prompt_tokens}")

# 3) estimate cost
# Assuming all prompt tokens are input and cached input tokens, and max_tokens are output tokens
input_costs = costs[model_name]["input"] * prompt_tokens
print(f"Estimated input cost: ${input_costs:.6f}")
cached_input_costs = costs[model_name]["cached_input"] * prompt_tokens
print(f"Estimated cached input cost: ${cached_input_costs:.6f}")
output_costs = costs[model_name]["output"] * max_tokens
print(f"Estimated output cost: ${output_costs:.6f}")
print(
    f"Total max estimated cost: ${input_costs + cached_input_costs + output_costs:.6f}"
)

Prompt tokens: 283
Estimated input cost: $0.000708
Estimated cached input cost: $0.000354
Estimated output cost: $0.001280
Total max estimated cost: $0.002341


In [55]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Hello!",
        }
    ],
    max_completion_tokens=128,
    top_p=0.95,
    model=deployment_name,
)

In [56]:
print(response)

ChatCompletion(id='chatcmpl-DZbwYqtPGjSYplodf3zZ6kdbFIaU2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! 😊 How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}})], created=1777379706, model='gpt-4o-2024-11-20', object='chat.completion', service_tier='default', system_fingerprint='fp_af7f7349a4', usage=CompletionUsage(completion_tokens=11, prompt_tokens=9, total_tokens=20, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {'jailbreak': {'filtered': False, 'de

In [60]:
print(response.to_dict())

{'id': 'chatcmpl-DZbwYqtPGjSYplodf3zZ6kdbFIaU2', 'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': 'Hello! 😊 How can I assist you today?', 'refusal': None, 'role': 'assistant', 'annotations': []}, 'content_filter_results': {'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}}}], 'created': 1777379706, 'model': 'gpt-4o-2024-11-20', 'object': 'chat.completion', 'service_tier': 'default', 'system_fingerprint': 'fp_af7f7349a4', 'usage': {'completion_tokens': 11, 'prompt_tokens': 9, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'jailbreak': {'filtered': False, 'detected': False}}}]}


```
ChatCompletion(id='chatcmpl-DZZFQtM00dyhVqTaXZ2XqHnOdRB1r', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! 😊 How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}})], created=1777369344, model='gpt-4o-2024-11-20', object='chat.completion', service_tier='default', system_fingerprint='fp_af7f7349a4', usage=CompletionUsage(completion_tokens=11, prompt_tokens=9, total_tokens=20, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {'jailbreak': {'filtered': False, 'detected': False}}}])
```


In [57]:
# Extract used tokens from response
# Structure of the response is:
# usage=CompletionUsage(completion_tokens=11, prompt_tokens=9, total_tokens=20, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))
print(f"Actual prompt tokens used: {response.usage.prompt_tokens}")
print(
    f"Actual cached input tokens used: {response.usage.prompt_tokens_details.cached_tokens}"
)
print(f"Actual completion tokens used: {response.usage.completion_tokens}")
print(f"Actual total tokens used: {response.usage.total_tokens}")
# Calculate actual cost based on usage
actual_input_cost = costs[model_name]["input"] * response.usage.prompt_tokens
actual_cached_input_cost = (
    costs[model_name]["cached_input"]
    * response.usage.prompt_tokens_details.cached_tokens
)
actual_output_cost = costs[model_name]["output"] * response.usage.completion_tokens
print(f"Input cost: ${actual_input_cost:.7f}")
print(f"Cached input cost: ${actual_cached_input_cost:.7f}")
print(f"Output cost: ${actual_output_cost:.7f}")
print(
    f"Total actual cost: ${actual_input_cost + actual_cached_input_cost + actual_output_cost:.7f}"
)

Actual prompt tokens used: 9
Actual cached input tokens used: 0
Actual completion tokens used: 11
Actual total tokens used: 20
Input cost: $0.0000225
Cached input cost: $0.0000000
Output cost: $0.0001100
Total actual cost: $0.0001325


```
"_cost": {
      "prompt_cost_uncached": "0.00079000",
      "prompt_cost_cached": "0.00000000",
      "completion_cost": "0.00048000",
      "total_cost": "0.00127000"
    }
```


In [33]:
client.close()

### Batch request


In [25]:
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=api_key,
)

In [ ]:
# System prompt: enforce Estonian rewriting behaviour, case conditioning, and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.

Replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Preserve tense, number, case, agreement, capitalisation, punctuation, and word order as much as possible.
- Infer the token’s grammatical role and morphology from the sentence context.
- Generate 10 alternatives that fit the context and use only these cases: nominative, genitive, partitive, or additive/illative (both fit the same role in this task).
- If the token is a proper name, replace it with another plausible proper name that still fits the required case.
- Do not repeat the original token unless it is the only valid option.
"""

# Structured output format for Azure OpenAI chat completions.
# Structured Outputs require a top-level object, so the list is wrapped in a candidates field.
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "candidates",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "candidates": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 10,
                    "maxItems": 10,
                }
            },
            "required": ["candidates"],
            "additionalProperties": False,
        },
    },
}

# One-shot example to anchor the output format
one_shot_user = """Sentence: Ma näen <maja>."""

one_shot_assistant = json.dumps(
    {
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "häärberit",
        ],
    },
    ensure_ascii=False,
    indent=2,
)

# Example real input sentence
# user_prompt = """Sentence: Samas on kõik uue kodu lähistel asuvad koolid sellised, mis võtavad <gümnaasiumi> vastu katsetega."""


# Create a user prompt template for the real sentences, using the same format as the few-shot example
def create_user_prompt(sentence, word_span):
    # Mark the target word with angle brackets
    start, end = word_span
    marked_sentence = (
        sentence[:start] + "<" + sentence[start:end] + ">" + sentence[end:]
    )
    return f"""Sentence: {marked_sentence}"""


def build_messages(user_prompt: str) -> list[dict[str, str]]:
    """Build the chat-completions message list for one rewrite request."""
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": one_shot_user},
        {"role": "assistant", "content": one_shot_assistant},
        {"role": "user", "content": user_prompt},
    ]


def parse_response_content(content: str) -> dict[str, Any]:
    """Parse and validate the JSON payload returned by Azure OpenAI."""
    parsed = json.loads(content or "{}")
    if not isinstance(parsed, dict):
        raise ValueError("Expected a JSON object from the model response.")
    if "candidates" not in parsed or not isinstance(parsed["candidates"], list):
        raise ValueError("Model response did not contain a candidates list.")
    return parsed


def append_result_to_json(file_path, record, id=None, search_id="sentence_id"):
    """Append one record to a JSON array stored in file_path."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    # Ensure any structured cost object is converted into plain JSON-serialisable data.
    if isinstance(record, dict) and "_cost" in record:
        cost_value = record["_cost"]
        if hasattr(cost_value, "as_dict"):
            record = dict(record)
            record["_cost"] = cost_value.as_dict()
    # If the file exists and is non-empty, load the existing data; otherwise start with an empty list
    if file_path.exists() and file_path.stat().st_size > 0:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = []
    else:
        data = []

    if not isinstance(data, list):
        data = []
    if id is not None and search_id is not None:
        # If ID is provided, check if a record with the same ID already exists and update it; otherwise append the new record
        existing_record = next(
            (r for r in data if int(r.get(search_id)) == int(id)), None
        )
        if existing_record:
            existing_record.update(record)
            # Remove error field if it exists, since we have a successful rewrite now
            existing_record.pop("error", None)
        else:
            data.append(
                record
            )  # Append the new record if no existing record with the same ID is found
    else:
        # If no ID is provided, put the record at the end of the list
        data.append(record)
    # Write the updated list back to the file with pretty formatting
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def rewrite_sentence_with_azure_openai(sentence_id, sentence, word_span, metadata):
    """Rewrite one sentence by replacing the marked target word via Azure OpenAI."""
    user_prompt = create_user_prompt(sentence=sentence, word_span=word_span)
    messages = build_messages(user_prompt)

    # Request a structured JSON response from Azure OpenAI chat completions.
    response = client.chat.completions.create(
        model=deployment_name,
        messages=messages,
        response_format=response_format,
        max_completion_tokens=1000,
        top_p=0.95,
    )
    # Estimate cost for this response (best-effort). The estimator expects the full response object.
    try:
        cost_info = estimate_cost_typed(response)
    except Exception as _err:
        # Don't fail the rewrite on estimator errors; keep a surfacable error message.
        cost_info = {"error": str(_err)}
    message = response.choices[0].message
    refusal = getattr(message, "refusal", None)
    if refusal:
        raise RuntimeError(f"Model refused the request: {refusal}")

    # Parse the JSON response and enrich it with metadata for later analysis.
    parsed = parse_response_content(message.content or "{}")

    # Keep useful metadata for later analysis/debugging.
    parsed["sentence_id"] = str(sentence_id)
    parsed["source_sentence"] = sentence
    parsed["target_word"] = metadata.get("target_word", "")
    if isinstance(word_span, list):
        parsed["word_span"] = word_span
    else:
        parsed["word_span"] = word_span.astype(
            int
        ).tolist()  # Convert numpy array to list for JSON serialization
    parsed["label"] = metadata.get("label", [])
    # Attach cost estimation metadata (not part of the structured schema)
    parsed["_cost"] = cost_info

    return parsed, response


# Backwards-compatible alias for any existing callers.
rewrite_sentence_with_genai = rewrite_sentence_with_azure_openai


def get_latest_processed_id(output_file):
    """Get the highest sentence ID that has already been processed and saved in the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        with open(output_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list) and len(data) > 0:
                return max(int(record["id"]) for record in data if "id" in record)
    return -1  # No valid records found, start from the beginning


def get_unprocessed_dataset(dataset, output_file):
    """Filter the dataset to include only records that have errors or have not been processed yet, based on the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        unprocessed_records = []
        processed_records = []
        # First, get the list of sentences that have errors in the output file
        with open(output_file, "r", encoding="utf-8") as f:
            processed_data = json.load(f)
            for record in processed_data:
                if "error" in record:
                    unprocessed_records.append(record["source_sentence"])
                else:
                    processed_records.append(record["source_sentence"])
        # Now filter the original dataset to include only records that are either unprocessed or have errors
        filtered_dataset = dataset[
            dataset["sentence"].isin(unprocessed_records)
            | ~dataset["sentence"].isin(processed_records)
        ]
        return filtered_dataset
    else:
        return dataset  # If no output file exists, return the entire dataset as unprocessed


def is_transient_error(exc: Exception) -> bool:
    """Heuristically detect retryable errors such as rate limit and network hiccups."""
    message = str(exc).lower()
    transient_markers = [
        "429",
        "rate",
        "quota",
        "too many requests",
        "timeout",
        "timed out",
        "connection",
        "temporar",
        "unavailable",
        "503",
        "502",
        "504",
        "internal",
    ]
    return any(marker in message for marker in transient_markers)


def rewrite_with_retry(sentence_id, sentence, word_span, metadata, config):
    """Call rewrite function with exponential backoff on transient errors."""
    for attempt in range(1, config["max_attempts"] + 1):
        try:
            return rewrite_sentence_with_azure_openai(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                metadata=metadata,
            )
        except Exception as exc:
            # should_retry = is_transient_error(exc) and attempt < config["max_attempts"]
            should_retry = (
                attempt < config["max_attempts"]
            )  # Retry on all exceptions for robustness, but still limit the number of attempts
            if not should_retry:
                raise

            backoff = min(
                config["max_backoff_seconds"],
                config["initial_backoff_seconds"] * (2 ** (attempt - 1)),
            )
            sleep_s = backoff + random.uniform(0.0, config["jitter_seconds"])
            print(
                f"Retry {attempt}/{config['max_attempts'] - 1} for sentence_id={sentence_id} "
                f"after transient error: {exc}. Sleeping {sleep_s:.2f}s..."
            )
            time.sleep(sleep_s)


In [32]:
# Load the homonyms dataset to get the sentences we want to rewrite with Gen AI
homonyms_df = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_updated_sentences.parquet"
)
display(homonyms_df.head())

,num,inflection_type,sentence,word,word_span,label,source
0,1,1,Edinburghi agulite mehe Irvine Welshi ja Glasg...,võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) o...",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoi...",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama mo...",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...
4,1,1,Itaalia president ütles Venemaa riigipea auks ...,Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-0...


In [ ]:
output_file = OUTPUT_DIR / "gpt-4o-homonyms_LLM_MLM.json"  # Output file for results
output_file_log = (
    OUTPUT_DIR / "gpt-4o-homonyms_LLM_MLM_log.json"
)  # More detailed log file for debugging and analysis
sample_df = homonyms_df.iloc[:1].copy()  # Use the first sentence for testing
sample_df["index"] = sample_df.index.astype(
    int
)  # Ensure index is an integer for ID purposes

# gpt-4o has following rate limits:
# - 180 000 tokens per minute (TPM)
# - 1080 requests per minute (RPM)
config = {
    # Pacing configuration to avoid hitting rate limits
    "base_delay_seconds": round(
        60 / 1080, 2
    ),  # RPM is 15 for gemini-3.1-flash-lite-preview
    "jitter_seconds": 0.1,  # Add a small random jitter to avoid thundering herd issues
    # Retry configuration for transient failures
    "max_attempts": 6,
    "initial_backoff_seconds": 2.0,
    "max_backoff_seconds": 60.0,
}

# Start from the next unprocessed sentence based on the output file contents
unprocessed_df = get_unprocessed_dataset(sample_df, output_file)
if len(unprocessed_df) == 0:
    print("No unprocessed sentences found. All done!")
else:
    print(f"Total unprocessed sentences to process: {len(unprocessed_df)}")
    print("Next sentence to process:")
    display(unprocessed_df.head(1))

Total unprocessed sentences to process: 1
Next sentence to process:


,num,inflection_type,sentence,word,word_span,label,source,index
0,1,1,Edinburghi agulite mehe Irvine Welshi ja Glasg...,võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...,0


In [34]:
# Find processed records that are in the sample_df
if output_file.exists() and output_file.stat().st_size > 0:
    with open(output_file, "r", encoding="utf-8") as f:
        processed_data = json.load(f)
        processed_ids = [
            int(record["id"]) for record in processed_data if "id" in record
        ]
        processed_sentences = sample_df[sample_df["index"].isin(processed_ids)]
        print(
            f"Number of sentences in sample_df that have already been processed: {len(processed_sentences)}"
        )
        print("Sample of processed sentences:")
        display(sample_df[sample_df["index"].isin(processed_ids)].head(10))
else:
    print("No output file found, so no sentences have been processed yet.")

# Now remove all the already processed sentences from the output file that are not in the sample_df
if output_file.exists() and output_file.stat().st_size > 0:
    with open(output_file, "r", encoding="utf-8") as f:
        processed_data = json.load(f)
        processed_ids = [
            int(record["id"]) for record in processed_data if "id" in record
        ]
        filtered_data = [
            record
            for record in processed_data
            if int(record.get("id", -1)) in sample_df["index"].values
        ]
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(filtered_data, f, ensure_ascii=False, indent=2)

No output file found, so no sentences have been processed yet.


In [ ]:
# Batch rewrite loop: call OpenAI and append each result to a JSON file
processed = 0
total_cost = 0.0
for i, (sentence_id, sentence, word_span, word, label) in enumerate(
    zip(
        unprocessed_df["index"],
        unprocessed_df["sentence"],
        unprocessed_df["word_span"],
        unprocessed_df["word"],
        unprocessed_df["label"],
    )
):
    try:
        metadata = {
            "target_word": word,
            "label": label.tolist(),  # Convert numpy array to list for JSON serialization
        }
        result, response = rewrite_with_retry(
            sentence_id=sentence_id,
            sentence=sentence,
            word_span=word_span,
            config=config,
            metadata=metadata,
        )
        append_result_to_json(
            output_file_log, response.to_dict(), id=sentence_id
        )  # Log the full response for debugging and analysis
        append_result_to_json(output_file, result, id=sentence_id)
        # Get the cost info from the result for logging
        cost_info = result.get("_cost", {})
        if "error" in cost_info:
            print(
                f"[{processed + 1}] Saved sentence_id={sentence_id} with cost estimation error: {cost_info['error']}"
            )
        else:
            total_cost_info = cost_info.get("total_cost", 0.0)
            total_cost += total_cost_info
            print(
                f"[{processed + 1}] Saved sentence_id={sentence_id} wth cost info: {total_cost_info:.6f}, total cost: {total_cost:.6f}"
            )
    except Exception as exc:
        error_record = {
            "id": str(sentence_id),
            "source_sentence": sentence,
            "target_word": word,
            "word_span": word_span.astype(int).tolist(),
            "label": label.tolist(),
            "error": str(exc),
        }
        append_result_to_json(output_file, error_record, id=sentence_id)
        print(f"[{processed + 1}] Error on sentence_id={sentence_id}: {exc}")

    processed += 1

    # Additional pacing between successful/failed items to avoid bursty traffic.
    if i < len(unprocessed_df) - 1:  # Don't sleep after the last item
        sleep_s = config["base_delay_seconds"] + random.uniform(
            0.0, config["jitter_seconds"]
        )
        print(f"Sleeping {sleep_s:.2f}s before next request...")
        time.sleep(sleep_s)

print(f"Finished. Processed {processed} rows. Results appended to: {output_file}")

[1] Error on sentence_id=0: Object of type CostBreakdown is not JSON serializable
Finished. Processed 1 rows. Results appended to: ..\outputs\gpt-4o-homonyms_LLM_MLM.json


In [47]:
result

{'candidates': ['laureaadi',
  'saaja',
  'tunnustatu',
  'auhinna',
  'pretendendi',
  'tiitli',
  'kandidaadi',
  'võistleja',
  'nominendi',
  'osaleja'],
 'id': '0',
 'source_sentence': 'Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.',
 'target_word': 'võitja',
 'word_span': [74, 80],
 'label': ['sg n'],
 '_cost': CostBreakdown(prompt_cost_uncached=Decimal('0.0007900'), prompt_cost_cached=Decimal('0.00'), completion_cost=Decimal('0.0004800'), total_cost=Decimal('0.0012700'))}

In [49]:
append_result_to_json(output_file, result, id=sentence_id)

In [36]:
client.close()